# 03. Extended bootstrap -- 5-day windows (Table A.16)

Bootstrap the mean off-diagonal ARI over random 5-day calendar windows,
holding the GMM regime fits fixed at the full-sample boundary. Tests
whether the dissonance signal in the stress / calm reference windows is
unusual relative to a random-window null.

The notebook runs `n_boot=100` in-process on `CL` (~10-20 s);
the cached CSV preserves the paper's `n_boot=1000` result.

**Re-run command**: `python run.py extended_bootstrap`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- load raw 5m OHLC for `CL`

Calls `src.data.data_ib.load_5m_ohlc` directly on `data/CL_5m.csv`. This is the same loader the pipeline uses; it parses mixed-offset DST timestamps as UTC then converts to NY local time.

In [2]:
from src.data.data_ib import load_5m_ohlc

FOCAL = "CL"
df_5m = load_5m_ohlc(DATA / f"{FOCAL}_5m.csv")
print(f"{FOCAL}_5m: {len(df_5m):,} bars, "
      f"{df_5m.index.min()} -> {df_5m.index.max()}, "
      f"{df_5m.index.normalize().nunique()} trading days")
df_5m.head()

CL_5m: 34,896 bars, 2025-11-02 18:00:00-05:00 -> 2026-05-01 16:55:00-04:00, 155 trading days


,Open,High,Low,Close,Volume
Date,,,,,
2025-11-02 18:00:00-05:00,58.36,58.43,58.10,58.29,2872.0
2025-11-02 18:05:00-05:00,58.30,58.37,58.29,58.35,669.0
2025-11-02 18:10:00-05:00,58.35,58.36,58.29,58.30,367.0
2025-11-02 18:15:00-05:00,58.30,58.33,58.26,58.33,394.0
2025-11-02 18:20:00-05:00,58.33,58.33,58.30,58.31,313.0


## Step 2 -- run a small bootstrap in-process

`src.experiments.exp_02_bootstrap.bootstrap_five_day_windows` fits GMM
regimes once on the full sample (frozen boundary), then resamples 5-day
windows uniformly without replacement and recomputes the mean off-diag
ARI on each window. Returns the calm and stress window ARIs together with
the bootstrap distribution and a two-sided p-value of `|calm - boot_median|`
against the bootstrap null.

In [3]:
from src.experiments.exp_02_bootstrap import bootstrap_five_day_windows
from src.core.config import EPISODES

stress_2026, calm_2026 = EPISODES["2026_iran"]
res = bootstrap_five_day_windows(
    df_5m, FOCAL,
    calm_window=calm_2026, stress_window=stress_2026,
    n_boot=100, seed=42,
)
pd.DataFrame([res])

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Win

,calm_ari,stress_ari,n_boot,boot_mean,boot_median,boot_q025,boot_q975,p_calm_vs_boot,p_stress_vs_boot
0,0.136,0.07,100,0.191,0.163,0.016,0.5,0.74,0.28


## Cached full-scale bootstrap (n_boot=1000) -- `outputs/bootstrap_five_day_windows.csv`

In [4]:
p = OUT / "bootstrap_five_day_windows.csv"
display(pd.read_csv(p).round(3) if p.exists() else Markdown(f"`{p}` missing -- run: `python run.py extended_bootstrap`"))

,calm_ari,stress_ari,n_boot,boot_mean,boot_median,boot_q025,boot_q975,p_calm_vs_boot,p_stress_vs_boot,symbol
0,0.097,0.190,1000,0.126,0.109,0.063,0.265,0.779,0.158,SPY
1,0.079,0.052,1000,0.115,0.081,-0.006,0.455,0.962,0.553,USDJPY
2,0.136,0.070,1000,0.201,0.164,-0.007,0.500,0.786,0.362,CL
3,0.147,0.143,1000,0.158,0.131,0.042,0.300,0.755,0.814,GLD
